In [1]:
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import random
from itertools import product


from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search

In [2]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

In [3]:
# set seed
random.seed(42)

# select a response variable (either marekt return or sp500 return)
y = response_variables['sprtrn']  # or 'sprtrn' for SP500 returns or vwretx

# transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# User-selected number of each 
num_topics = len(topic_cols)  
num_stocks = 0

# Randomly sample (without replacement), limited by available count
selected_topics = random.sample(topic_cols, min(num_topics, len(topic_cols)))
selected_stocks = random.sample(stock_cols, min(num_stocks, len(stock_cols)))

# Final filtered dataframe
X = X[selected_topics + selected_stocks]

# Convert stock returns to log returns: log(1+r)
X[selected_stocks] = np.log(X[selected_stocks] + 1)

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

## Grid Search

In [4]:
# Objective Function
def objective_function(row):
    if 1.96 < row['kappa_tstat'] <= 20:
        return row['r2_insample_stage2'] * row['kappa']
    else:
        return -float('inf')

# -----------------------------
# Stopping criteria (NEW)
# -----------------------------
max_iterations = 20          # maximum refinement iterations
improvement_tol = 1e-6       # stop if best objective improves by less than this

# Initial grid
window_sizes = [10, 50, 100, 200, 300, 400, 500]
n_lags = [1,2,3,4,5,6,7,8,9,10]
lambda_base = 0.01
lambda_values = [lambda_base * factor for factor in [0.8, 0.9, 1.0, 1.1, 1.2]]

# IMPORTANT: match grid_search expectations: window_sizes, n_lags, lambdas
current_param_grid = {
    'window_sizes': window_sizes,
    'n_lags': n_lags,
    'lambdas': lambda_values
}

results_all_rounds = []

prev_best_objective = -float('inf')

# ----------------------------------------
# loop 
# ----------------------------------------
for i in range(max_iterations):

    print(f"\nStarting refinement iteration {i+1}")

    # Run grid search
    summary_df, coefficients_df = grid_search(
        X, y, current_param_grid, verbose=True
    )

    # Evaluate objective
    summary_df['objective'] = summary_df.apply(objective_function, axis=1)

    # Track results
    results_all_rounds.append(summary_df.copy())

    # Pick best point of this iteration
    best_idx = summary_df['objective'].idxmax()
    best_row = summary_df.loc[best_idx]
    best_objective = float(best_row['objective'])

    print("Best this iteration:")
    print(best_row[['window_size', 'n_lags', 'lambda', 'objective']])

    # Early stopping checks
    if not np.isfinite(best_objective):
        print("Stopping: best objective is not finite (all candidates failed constraints).")
        break

    if i > 0:
        improvement = best_objective - prev_best_objective
        print(f"Improvement vs previous best: {improvement:.6g}")

        if improvement < improvement_tol:
            print(f"Stopping: improvement {improvement:.6g} < tolerance {improvement_tol:.6g}.")
            break

    prev_best_objective = best_objective

    # ----------------------------------------
    # Build new grid around best point
    # ----------------------------------------
    best_window = int(best_row['window_size'])
    best_n_lags = int(best_row['n_lags'])
    best_lambda = float(best_row['lambda'])

    # shrink window search range
    window_sizes_refined = list(range(best_window - 25, best_window + 26, 5))
    window_sizes_refined = [w for w in window_sizes_refined if w > 20]

    # shrink n_lags search range
    n_lags_refined = list(range(best_n_lags - 1, best_n_lags + 2))
    n_lags_refined = [l for l in n_lags_refined if l >= 1]

    # shrink lambda range 
    lambda_values_refined = np.linspace(
        0.8 * best_lambda,
        1.2 * best_lambda,
        num=10
    )

    # update grid for next iteration
    current_param_grid = {
        'window_sizes': window_sizes_refined,
        'n_lags': n_lags_refined,
        'lambdas': lambda_values_refined
    }

# ----------------------------------------
# Final evaluation
final_df = pd.concat(results_all_rounds, ignore_index=True)
final_df['objective'] = final_df.apply(objective_function, axis=1)
best_overall = final_df.loc[final_df['objective'].idxmax()]

print("\nBest hyperparameters after refinement:")
print(best_overall[['window_size', 'n_lags', 'lambda', 'objective']])


Starting refinement iteration 1
Testing 350 configurations...


Grid search: 100%|██████████| 350/350 [15:52<00:00,  2.72s/it]
c:\Users\jonat\Lasso_paper\Empirical\scripts\lasso_11_2025\grid_search.py:197: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  coefficients_df = pd.concat(details_list, ignore_index=True)



GRID SEARCH COMPLETE
Best this iteration:
window_size    50.000000
n_lags          4.000000
lambda          0.009000
objective       0.001716
Name: 66, dtype: float64

Starting refinement iteration 2
Testing 330 configurations...


Grid search: 100%|██████████| 330/330 [05:04<00:00,  1.08it/s]
c:\Users\jonat\Lasso_paper\Empirical\scripts\lasso_11_2025\grid_search.py:197: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  coefficients_df = pd.concat(details_list, ignore_index=True)



GRID SEARCH COMPLETE
Best this iteration:
window_size    45.000000
n_lags          4.000000
lambda          0.009200
objective       0.006567
Name: 135, dtype: float64
Improvement vs previous best: 0.00485124

Starting refinement iteration 3
Testing 300 configurations...


Grid search: 100%|██████████| 300/300 [04:59<00:00,  1.00it/s]
c:\Users\jonat\Lasso_paper\Empirical\scripts\lasso_11_2025\grid_search.py:197: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  coefficients_df = pd.concat(details_list, ignore_index=True)



GRID SEARCH COMPLETE
Best this iteration:
window_size    45.000000
n_lags          4.000000
lambda          0.009404
objective       0.005892
Name: 135, dtype: float64
Improvement vs previous best: -0.000675094
Stopping: improvement -0.000675094 < tolerance 1e-06.

Best hyperparameters after refinement:
window_size    45.000000
n_lags          4.000000
lambda          0.009200
objective       0.006567
Name: 498, dtype: float64


In [10]:
# print for the 5 rows with the highest objective values r2 insample_stage2 and stage1, kappa ,kappa_tstat, lambda, window_size, n_lags
top_10 = final_df.nlargest(10, 'objective')
print(top_10[['r2_insample_stage2', 'r2_insample_stage1', 'r2_oos_stage2', 'kappa', 'kappa_tstat', 'lambda', 'window_size', 'n_lags']])

     r2_insample_stage2  r2_insample_stage1  r2_oos_stage2     kappa  \
27             0.005648            0.884429       0.004034  0.066512   
2              0.005524            0.884621       0.006996  0.064939   
107            0.000644            0.002039      -0.000491  0.490569   
0              0.003790            0.884312       0.010028  0.054524   
34             0.003501            0.871994       0.003859  0.054591   
3              0.003148            0.884322       0.006735  0.050145   
4              0.003097            0.871156       0.006165  0.049188   
13             0.003048            0.871362       0.004967  0.049475   
39             0.002895            0.851988       0.003791  0.050978   
18             0.002972            0.850828       0.004550  0.049390   

     kappa_tstat  lambda  window_size  n_lags  
27      3.481757  0.0001           10       9  
2       3.437302  0.0001           10      10  
107     1.981315  0.0020          300       3  
0       2.81538

## Bayesian Optimization with optuna

In [28]:
import optuna
import numpy as np

def objective(trial):
    # 1) Suggest parameters
    window_size = trial.suggest_int("window_size", 10, 400)
    n_lags = trial.suggest_int("n_lags", 1, 15)
    lam = trial.suggest_float("lambda", 1e-5, 1e-1, log=True)

    # 2) Adapt to grid_search API
    current_param_grid = {
        "window_sizes": [window_size],
        "n_lags": [n_lags],
        "lambdas": [lam],
    }

    # 3) Run model
    summary_df, _ = grid_search(X, y, current_param_grid, verbose=False)

    # Hard failure → prune
    if summary_df is None or summary_df.empty:
        raise optuna.TrialPruned()

    row = summary_df.iloc[0]

    # 4) Safely extract components
    r2 = row.get("r2_insample_stage2", 0.0)
    kappa = row.get("kappa", 0.0)


    # 5) New objective: explicitly penalizes low / zero t-stats
    objective_value = r2 * kappa 

    return float(objective_value)

# -----------------------------
# Run the Optimization
# -----------------------------
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=150)

print("\nBest Parameters found:")
print(study.best_params)
print(f"Best Objective Value: {study.best_value}")

[I 2026-01-07 17:04:46,430] A new study created in memory with name: no-name-8afa3f49-4c82-461e-8c3f-9bc464504bc5
[I 2026-01-07 17:04:59,076] Trial 0 finished with value: -3.905296086514909e-10 and parameters: {'window_size': 89, 'n_lags': 13, 'lambda': 0.02225002788005624}. Best is trial 0 with value: -3.905296086514909e-10.
[I 2026-01-07 17:10:38,765] Trial 1 finished with value: -2.250928556699267e-15 and parameters: {'window_size': 216, 'n_lags': 12, 'lambda': 1.763542395502356e-05}. Best is trial 1 with value: -2.250928556699267e-15.
[I 2026-01-07 17:10:42,211] Trial 2 finished with value: -3.198115883229491e-09 and parameters: {'window_size': 277, 'n_lags': 1, 'lambda': 0.02467193372182004}. Best is trial 1 with value: -2.250928556699267e-15.
[I 2026-01-07 17:12:40,461] Trial 3 finished with value: -6.805711869963132e-15 and parameters: {'window_size': 88, 'n_lags': 7, 'lambda': 3.796841493587753e-05}. Best is trial 1 with value: -2.250928556699267e-15.
[I 2026-01-07 17:14:09,317


Best Parameters found:
{'window_size': 30, 'n_lags': 10, 'lambda': 0.014865485692861374}
Best Objective Value: 0.0038997741732938788
